# SIEWS+ 5.0 — Stage 3: Fire & Smoke Detection
## Transfer Learning with YOLO26n

**Target classes:**
- `0: fire`
- `1: smoke`

**Strategy:** Fine-tune pretrained YOLO26n dari COCO dengan dataset Fire & Smoke khusus.

---
**Cara pakai di Kaggle:**
1. Upload notebook ini ke Kaggle
2. Add dataset: `Fire and Smoke.v14i.yolo26` (upload dari folder `dataset/`)
3. Enable GPU P100/T4
4. Run All → download `best_stage3_new.pt`

## 1. Setup Environment

In [ ]:
import os
import subprocess

# Detect environment
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = 'google.colab' in str(globals().get('__builtins__', ''))
print(f'Kaggle: {IS_KAGGLE} | Colab: {IS_COLAB}')

# Install ultralytics
subprocess.run(['pip', 'install', '-U', 'ultralytics', '-q'], check=True)

import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = '0'
else:
    print('No GPU — using CPU (akan sangat lambat!)')
    DEVICE = 'cpu'

## 2. Konfigurasi Path

In [ ]:
from pathlib import Path

# =====================================================
# SESUAIKAN path dataset di sini
# =====================================================
if IS_KAGGLE:
    OUTPUT_DIR = Path('/kaggle/working')

    # ── Langkah 1: Cari ZIP ──────────────────────────────────────────────────
    _zip_candidates = list(Path('/kaggle/input').rglob('*.zip'))
    _fire_zips = [p for p in _zip_candidates
                  if 'fire' in p.name.lower() or 'smoke' in p.name.lower()]
    STAGE3_ZIP = _fire_zips[0] if _fire_zips else (_zip_candidates[0] if _zip_candidates else None)

    # ── Langkah 2: Cari folder dataset (train/images) langsung ───────────────
    # Kaggle bisa attach dataset sebagai folder (tanpa zip), terutama dari Roboflow.
    # Cari parent folder yang punya struktur train/ + (valid/ atau val/)
    _found_train_dirs = list(Path('/kaggle/input').rglob('train/images'))
    if _found_train_dirs:
        # Ambil yang paling pendek path-nya (paling root)
        _found_train_dirs.sort(key=lambda p: len(p.parts))
        _kaggle_dataset_dir = _found_train_dirs[0].parent.parent  # .../train/images → parent 2x
        print(f'Dataset folder ditemukan: {_kaggle_dataset_dir}')
    else:
        _kaggle_dataset_dir = None

    if STAGE3_ZIP:
        print(f'ZIP ditemukan: {STAGE3_ZIP}')
    if _kaggle_dataset_dir is None and STAGE3_ZIP is None:
        print('WARNING: Tidak ada ZIP maupun folder dataset di /kaggle/input/')
        print('  → Pastikan dataset sudah di-attach: Notebook → Data → Add Input')

else:
    # Lokal (Windows)
    OUTPUT_DIR        = Path('F:/migas-siews/training/runs')
    DATASET_ROOT      = Path('F:/migas-siews/dataset')
    _fire_zips        = list(DATASET_ROOT.rglob('*.zip'))
    STAGE3_ZIP        = _fire_zips[0] if _fire_zips else DATASET_ROOT / 'Fire and Smoke.v14i.yolo26.zip'
    _kaggle_dataset_dir = None  # tidak relevan di lokal

EXTRACT_DIR = OUTPUT_DIR / 'stage3_dataset'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'STAGE3_ZIP  : {STAGE3_ZIP}')
print(f'EXTRACT_DIR : {EXTRACT_DIR}')


## 3. Ekstrak & Validasi Dataset

In [ ]:
import zipfile, shutil

# ── Resolve EXTRACT_DIR: hapus symlink lama jika ada ─────────────────────────
if EXTRACT_DIR.is_symlink():
    EXTRACT_DIR.unlink()   # hapus symlink lama (bukan folder aslinya)
    print(f'Symlink lama dihapus: {EXTRACT_DIR}')

if EXTRACT_DIR.exists():
    # Folder nyata sudah ada (hasil extract sebelumnya)
    print(f'Already extracted: {EXTRACT_DIR}')

elif IS_KAGGLE and _kaggle_dataset_dir is not None:
    # ── Kasus: Dataset sudah sebagai folder di /kaggle/input ─────────────────
    print(f'Dataset folder mode (no ZIP).')
    print(f'  Source : {_kaggle_dataset_dir}')
    print(f'  Symlink: {EXTRACT_DIR}')
    EXTRACT_DIR.symlink_to(_kaggle_dataset_dir)
    print('Symlink created!')

elif STAGE3_ZIP is not None and STAGE3_ZIP.exists():
    # ── Kasus: Ada ZIP → extract ──────────────────────────────────────────────
    print(f'Extracting {STAGE3_ZIP.name} ...')
    with zipfile.ZipFile(STAGE3_ZIP, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print('Extracted!')

else:
    raise FileNotFoundError(
        'Dataset tidak ditemukan!\n'
        'Solusi di Kaggle:\n'
        '  1. Klik "+ Add Input" di panel kanan\n'
        '  2. Cari dataset: "fire smoke detection yolo"\n'
        '  3. Pastikan folder train/ dan valid/ ada di dataset'
    )

# Verifikasi struktur dataset
print('\nStruktur dataset:')
for split in ['train', 'valid', 'val', 'test']:
    img_dir = EXTRACT_DIR / split / 'images'
    lbl_dir = EXTRACT_DIR / split / 'labels'
    if img_dir.exists():
        n_img = len(list(img_dir.glob('*.*')))
        n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        print(f'  [{split:6s}] images={n_img:5d}  labels={n_lbl:5d}')

yaml_found = sorted(EXTRACT_DIR.rglob('*.yaml'))
if yaml_found:
    print('\nYAML files:')
    for y in yaml_found:
        print(f'  {y}')
else:
    print('\nINFO: Tidak ada .yaml → akan dibuat di Step 4.')


In [ ]:
# Cari data.yaml — hanya untuk preview isi dataset asli.
# YAML yang dipakai training adalah custom_yaml yang dibuat di Step 4.
yaml_files = list(EXTRACT_DIR.rglob('data.yaml'))
if not yaml_files:
    yaml_files = list(EXTRACT_DIR.rglob('*.yaml'))

if yaml_files:
    DATA_YAML = yaml_files[0]
    print(f'YAML ditemukan: {DATA_YAML}')
    print('\n--- YAML content (original dari dataset) ---')
    print(DATA_YAML.read_text())
else:
    DATA_YAML = None
    print('INFO: Tidak ada .yaml di dataset (normal untuk beberapa format Roboflow).')
    print('      Custom YAML akan dibuat di Step 4 — tidak perlu khawatir.')


In [ ]:
# Hitung jumlah gambar per split
for split in ['train', 'valid', 'val', 'test']:
    img_dir = EXTRACT_DIR / split / 'images'
    if img_dir.exists():
        imgs = list(img_dir.glob('*.*'))
        lbl_dir = EXTRACT_DIR / split / 'labels'
        lbls = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
        print(f'[{split:6s}] images={len(imgs):5d}  labels={len(lbls):5d}')

## 4. Buat YAML Custom (Fix Path) + Label Audit

> **Kritis:** Cell ini juga menjalankan **label audit** — verifikasi bahwa `0=fire` dan `1=smoke` di file `.txt` dataset. Jika ada WARNING, jalankan cell *DEBUG audit visual* di bawahnya.

In [ ]:
import yaml
from collections import Counter

# Deteksi nama split val (bisa 'valid' atau 'val')
val_split = 'valid' if (EXTRACT_DIR / 'valid' / 'images').exists() else 'val'

# ─── SATU SUMBER KEBENARAN ───────────────────────────────────────────────────
# Roboflow export dengan urutan alfabetis: fire (0) < smoke (1).
# JANGAN ubah urutan ini tanpa audit label di bawah.
CANONICAL_CLASS_NAMES  = {0: 'fire', 1: 'smoke'}           # dict: untuk lookup di kode
CANONICAL_CLASS_LIST   = ['fire', 'smoke']                  # list: untuk YAML (wajib!)
CANONICAL_CLASS_COLORS = {0: (255, 80, 0), 1: (180, 180, 180)}  # RGB: fire=oranye, smoke=abu

# ─── LABEL AUDIT: verifikasi class distribution di train set ─────────────────
print('--- Label Audit ---')
train_lbl_dir = EXTRACT_DIR / 'train' / 'labels'
cls_counter = Counter()
n_files_audited = 0
for lbl_file in train_lbl_dir.glob('*.txt'):
    for line in lbl_file.read_text().strip().splitlines():
        parts = line.split()
        if parts:
            cls_counter[int(parts[0])] += 1
    n_files_audited += 1

print(f'Files audited : {n_files_audited}')
for cls_id, count in sorted(cls_counter.items()):
    name = CANONICAL_CLASS_NAMES.get(cls_id, f'UNKNOWN-{cls_id}')
    print(f'  cls {cls_id} ({name:6s}): {count:6d} bounding boxes')

# Sanity check: harus ada tepat 2 class (0 dan 1)
unexpected = [k for k in cls_counter if k not in CANONICAL_CLASS_NAMES]
if unexpected:
    print(f'\nWARNING: Ada class ID yang tidak dikenali: {unexpected}')
    print('         Periksa kembali dataset — mungkin ada label rusak.')
else:
    print('\n✅ Class ID 0 dan 1 semua valid.')

# ─── BUAT YAML DENGAN `names` SEBAGAI LIST ───────────────────────────────────
# KRITIS: YOLO membaca `names` sebagai list (index = class ID).
# Menggunakan dict dengan integer key akan menyebabkan class mapping terbalik/error.
custom_yaml = OUTPUT_DIR / 'stage3_fire_smoke.yaml'
yaml_content = {
    'path': str(EXTRACT_DIR),
    'train': 'train/images',
    'val'  : f'{val_split}/images',
    'test' : 'test/images' if (EXTRACT_DIR / 'test' / 'images').exists() else f'{val_split}/images',
    'nc'   : 2,
    'names': CANONICAL_CLASS_LIST,   # <-- LIST, bukan dict!
}

with open(custom_yaml, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)

print(f'\nCustom YAML: {custom_yaml}')
print(custom_yaml.read_text())

# Verifikasi: parse balik YAML dan cek names
_verify = yaml.safe_load(custom_yaml.read_text())
assert _verify['names'] == CANONICAL_CLASS_LIST, f'YAML names mismatch: {_verify["names"]}' 
assert _verify['nc']    == 2, f'YAML nc mismatch: {_verify["nc"]}'
print('✅ YAML verified: names =', _verify['names'])


In [ ]:
# ─── DEBUG: Audit Visual Label (jalankan jika fire/smoke masih terbalik) ─────
# Tampilkan 3 gambar DENGAN ground truth label untuk verifikasi manual.
# Jika kotak hijau "GT:fire" menunjuk ke asap → urutan class terbalik di dataset.

import cv2, random
import matplotlib.pyplot as plt

_audit_img_dir = EXTRACT_DIR / 'train' / 'images'
_audit_lbl_dir = EXTRACT_DIR / 'train' / 'labels'
_audit_samples = random.sample(
    list(_audit_img_dir.glob('*.jpg')) + list(_audit_img_dir.glob('*.png')),
    min(4, len(list(_audit_img_dir.glob('*.*'))))
)

fig, axes = plt.subplots(1, len(_audit_samples), figsize=(16, 5))
if len(_audit_samples) == 1:
    axes = [axes]

for ax, img_path in zip(axes, _audit_samples):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = _audit_lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            if not parts: continue
            cls_id = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = int((xc - bw/2) * w); y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w); y2 = int((yc + bh/2) * h)
            name = CANONICAL_CLASS_NAMES.get(cls_id, f'UNKNOWN-{cls_id}')
            # RGB warna
            color_rgb = CANONICAL_CLASS_COLORS[cls_id] if cls_id in CANONICAL_CLASS_COLORS else (0,255,0)
            import numpy as np
            cv2.rectangle(img, (x1,y1), (x2,y2), color_rgb, 3)
            cv2.putText(img, f'ID:{cls_id}={name}', (x1, max(y1-8,15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, color_rgb, 2)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(img_path.name[:25], fontsize=9)

plt.suptitle(
    'LABEL AUDIT — Verifikasi manual: apakah ID:0=fire dan ID:1=smoke BENAR?\n'
    'Oranye=fire, Abu=smoke. Jika terbalik → ubah CANONICAL_CLASS_NAMES.',
    fontsize=11, fontweight='bold', color='darkred'
)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'label_audit.png'), dpi=100)
plt.show()
print('Simpan screenshot ini untuk verifikasi class order.')


## 5. Visualisasi Sample Dataset

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

COLORS = CANONICAL_CLASS_COLORS
NAMES  = CANONICAL_CLASS_NAMES

train_img_dir = EXTRACT_DIR / 'train' / 'images'
train_lbl_dir = EXTRACT_DIR / 'train' / 'labels'
samples = random.sample(list(train_img_dir.glob('*.jpg')) + list(train_img_dir.glob('*.png')), min(6, len(list(train_img_dir.glob('*.*')))))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, samples):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = train_lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            cls, xc, yc, bw, bh = map(float, line.split())
            cls = int(cls)
            x1 = int((xc - bw/2) * w); y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w); y2 = int((yc + bh/2) * h)
            color = COLORS.get(cls, (0,255,0))
            cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
            cv2.putText(img, f'ID:{cls} {NAMES.get(cls, "?")}', (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    ax.imshow(img); ax.axis('off'); ax.set_title(img_path.name[:20])

plt.suptitle('Sample Dataset — Fire & Smoke', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'dataset_samples.png'), dpi=100)
plt.show()
print('Samples visualized!')

## 6. Load Base Model YOLO26n

In [ ]:
from ultralytics import YOLO

# Download pretrained YOLO26n (COCO, 80 classes)
# YOLO26 = end-to-end model tanpa NMS — lebih cepat inferensi
base_model = YOLO('yolo26n.pt')

print(f'Model loaded: {base_model.model_name}')
print(f'Base classes: {len(base_model.names)} (COCO)')
print('\nSiap untuk fine-tuning ke 2 kelas: fire, smoke')

## 7. Training (Transfer Learning)

### Penjelasan Hyperparameter:
| Parameter | Nilai | Alasan |
|---|---|---|
| `epochs` | 80 | Cukup untuk konvergensi fire/smoke |
| `batch` | 16 | Sesuai VRAM GPU Kaggle |
| `imgsz` | 640 | Resolusi standar YOLO |
| `patience` | 15 | Early stopping jika tidak improve |
| `freeze` | 10 | Freeze 10 layer pertama — transfer learning |
| `hsv_h/s/v` | tinggi | Fire/smoke sangat bergantung warna |
| `mosaic` | 1.0 | Gabungkan 4 gambar — perbanyak variasi |
| `flipud` | 0.1 | Api bisa dari segala arah |
| `optimizer` | `AdamW` | Lebih stabil untuk fine-tuning |

In [ ]:
# =====================================================
# KONFIGURASI TRAINING — sesuaikan jika perlu
# =====================================================
EPOCHS   = 80
BATCH    = 16   # kurangi ke 8 jika OOM (out of memory)
IMGSZ    = 640
PATIENCE = 15   # early stopping
FREEZE   = 10   # freeze backbone layers — transfer learning

print(f'Config: epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}, device={DEVICE}')
print(f'        freeze={FREEZE} layers (transfer learning mode)')

In [ ]:
results = base_model.train(
    data    = str(custom_yaml),
    epochs  = EPOCHS,
    batch   = BATCH,
    imgsz   = IMGSZ,
    device  = DEVICE,
    patience= PATIENCE,
    freeze  = FREEZE,          # transfer learning: bekukan backbone
    pretrained = True,

    # Project output
    project  = str(OUTPUT_DIR / 'runs'),
    name     = 'stage3_fire_smoke',
    exist_ok = True,

    # Augmentasi — penting untuk fire/smoke
    augment  = True,
    mosaic   = 1.0,          # 4-image mosaic
    mixup    = 0.1,          # mix 2 gambar
    copy_paste = 0.1,        # copy-paste objek ke gambar lain
    hsv_h    = 0.03,         # variasi hue (warna api berbeda-beda)
    hsv_s    = 0.9,          # variasi saturasi (asap tebal/tipis)
    hsv_v    = 0.5,          # variasi brightness (siang/malam)
    fliplr   = 0.5,          # flip horizontal
    flipud   = 0.1,          # flip vertikal (api dari berbagai sudut)
    degrees  = 5.0,          # rotasi kecil
    scale    = 0.6,          # variasi skala objek
    translate= 0.1,
    erasing  = 0.3,          # random erasing — robustness

    # Optimizer
    optimizer     = 'AdamW', # lebih stabil untuk fine-tuning
    lr0           = 0.001,   # learning rate awal — rendah untuk fine-tuning
    lrf           = 0.01,    # learning rate akhir (cosine decay)
    weight_decay  = 0.0005,
    warmup_epochs = 3,

    # Save & logging
    save_period = 10,
    verbose     = True,
    plots       = True,
)

## 8. Evaluasi Hasil Training

In [ ]:
# Tampilkan metrik training
metrics = results.results_dict

print('=' * 50)
print('HASIL TRAINING — Stage 3 Fire/Smoke')
print('=' * 50)
map50    = metrics.get('metrics/mAP50(B)', 0)
map5095  = metrics.get('metrics/mAP50-95(B)', 0)
prec     = metrics.get('metrics/precision(B)', 0)
rec      = metrics.get('metrics/recall(B)', 0)

print(f'mAP50       : {map50:.4f}  (target: > 0.85)')
print(f'mAP50-95    : {map5095:.4f}')
print(f'Precision   : {prec:.4f}  (target: > 0.80)')
print(f'Recall      : {rec:.4f}  (target: > 0.80)')
print()

if map50 >= 0.85:
    print('✅ Model sudah baik! mAP50 >= 0.85')
elif map50 >= 0.70:
    print('⚠️  Model cukup baik. Coba tambah epochs atau lebih banyak data.')
else:
    print('❌ Model kurang baik. Lihat saran di bawah.')

In [ ]:
# Tampilkan grafik training (loss curves, mAP)
run_dir = OUTPUT_DIR / 'runs' / 'stage3_fire_smoke'

plot_files = [
    run_dir / 'results.png',
    run_dir / 'confusion_matrix.png',
    run_dir / 'PR_curve.png',
]

from IPython.display import Image, display
for pf in plot_files:
    if pf.exists():
        print(f'--- {pf.name} ---')
        display(Image(str(pf), width=800))

## 9. Validasi pada Val Set

In [ ]:
# Load best model dan validasi
best_pt = run_dir / 'weights' / 'best.pt'
print(f'Loading best model: {best_pt}')

best_model = YOLO(str(best_pt))

# ─── Patch & Re-save: satu-satunya cara aman paksa metadata class ─────────────
# Hanya override Python attribute TIDAK cukup — predict() pakai internal C++ metadata.
# Solusi: patch → save ke disk → reload dari disk.
loaded_names = {int(k): v for k, v in dict(best_model.names).items()}
print(f'Model names (dari weight): {loaded_names}')

if loaded_names != CANONICAL_CLASS_NAMES:
    print(f'WARNING: Class order berbeda dari canonical! {loaded_names}')
    print(f'         → Melakukan patch dan re-save...')
    best_model.model.names = CANONICAL_CLASS_NAMES   # patch internal model
    patched_pt = run_dir / 'weights' / 'best_patched.pt'
    best_model.save(str(patched_pt))
    best_model = YOLO(str(patched_pt))               # reload dari disk
    print(f'Patched model saved & reloaded: {patched_pt}')
else:
    print(f'✅ Class order OK: {loaded_names}')

# Konfirmasi setelah patch
final_names = {int(k): v for k, v in dict(best_model.names).items()}
print(f'Final model.names: {final_names}')
assert final_names == CANONICAL_CLASS_NAMES, f'Patch gagal: {final_names}'

val_results = best_model.val(data=str(custom_yaml), device=DEVICE, verbose=True)

print(f'\nVal mAP50   : {val_results.box.map50:.4f}')
print(f'Val mAP50-95: {val_results.box.map:.4f}')


## 10. Inference Test — Uji Langsung pada Gambar

In [ ]:
# Ambil beberapa gambar test dan jalankan inferensi
# PENTING: pastikan best_model sudah di-patch di Step 9 sebelum menjalankan ini.
assert {int(k): v for k, v in dict(best_model.names).items()} == CANONICAL_CLASS_NAMES, \
    "best_model.names tidak sesuai CANONICAL! Jalankan ulang Step 9."

test_img_dir = EXTRACT_DIR / 'test' / 'images'
if not test_img_dir.exists():
    test_img_dir = EXTRACT_DIR / val_split / 'images'
test_lbl_dir = test_img_dir.parent.parent / test_img_dir.parent.name.replace('images', 'labels') \
    if False else test_img_dir.parent.parent / 'labels'  # noqa: simpel saja

# Gunakan semua gambar jika < 6, sample jika lebih
all_test = list(test_img_dir.glob('*.jpg')) + list(test_img_dir.glob('*.png'))
test_imgs = random.sample(all_test, min(6, len(all_test)))

TEST_CONF = 0.15  # smoke diffuse, conf rendah untuk audit visual

# ─── Warna dalam BGR (untuk cv2) ─────────────────────────────────────────────
# KRITIS: gunakan BGR, bukan RGB. Jangan mix dengan CANONICAL_CLASS_COLORS (RGB).
PRED_COLORS_BGR = {
    0: (0, 140, 255),    # fire  → oranye terang  (B=0, G=140, R=255)
    1: (180, 180, 180),  # smoke → abu-abu netral (B=G=R=180)
}
GT_COLOR_BGR    = (0, 255, 80)   # ground truth → hijau

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, test_imgs):
    pred    = best_model.predict(str(img_path), conf=TEST_CONF, verbose=False)[0]
    img_bgr = cv2.imread(str(img_path))
    h, w    = img_bgr.shape[:2]
    boxes   = pred.boxes
    label_parts = []

    # ── Draw ground truth (jika ada) dengan kotak hijau tipis ──
    lbl_path = test_img_dir.parent.parent / 'labels' / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            if not parts:
                continue
            cls_gt = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            gx1 = int((xc - bw/2) * w); gy1 = int((yc - bh/2) * h)
            gx2 = int((xc + bw/2) * w); gy2 = int((yc + bh/2) * h)
            gt_name = CANONICAL_CLASS_NAMES.get(cls_gt, f'cls_{cls_gt}')
            cv2.rectangle(img_bgr, (gx1, gy1), (gx2, gy2), GT_COLOR_BGR, 1)
            cv2.putText(img_bgr, f'GT:{gt_name}', (gx1, max(gy1 - 18, 14)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, GT_COLOR_BGR, 1)

    # ── Draw predictions ──────────────────────────────────────────────────────
    if boxes is not None and len(boxes) > 0:
        for box in boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            # Lookup nama dari CANONICAL — ini yang jadi sumber kebenaran label
            name  = CANONICAL_CLASS_NAMES.get(cls_id, f'UNKNOWN-{cls_id}')
            color = PRED_COLORS_BGR.get(cls_id, (0, 255, 0))

            cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img_bgr, f'{name} {conf:.0%}', (x1, max(y1 - 6, 18)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
            label_parts.append(f'ID:{cls_id}={name}({conf:.0%})')

    annotated = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.axis('off')
    preds_str = ', '.join(label_parts) if label_parts else '— no detection —'
    ax.set_title(preds_str[:40], fontsize=8, color='white' if not label_parts else 'black')

plt.suptitle(
    'Inference Test — Best Model\n(oranye=fire, abu=smoke, hijau tipis=GT)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'inference_test.png'), dpi=100)
plt.show()
print('\nInference test selesai! Cek apakah label fire/smoke sudah benar.')
print('Jika masih terbalik → audit label dataset dengan cell di bawah ini.')


## 11. Simpan Model ke Backend

In [ ]:
import shutil

best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'

# Simpan ulang best model dengan metadata class yang sudah dipaksa benar.
export_model = best_model if 'best_model' in globals() else YOLO(str(best_pt))
export_model.names = CANONICAL_CLASS_NAMES

if IS_KAGGLE:
    # Di Kaggle: salin ke /kaggle/working agar bisa di-download
    out_best = Path('/kaggle/working/best_stage3_new.pt')
    out_last = Path('/kaggle/working/last_stage3_new.pt')
    export_model.save(str(out_best))
    shutil.copy2(last_pt, out_last)
    print(f'✅ Saved (Kaggle output):')
    print(f'   {out_best}  ({out_best.stat().st_size/1e6:.1f} MB)')
    print(f'   {out_last}  ({out_last.stat().st_size/1e6:.1f} MB)')
    print()
    print('Langkah selanjutnya:')
    print('  1. Download best_stage3_new.pt dari Output tab Kaggle')
    print('  2. Salin ke: f:/migas-siews/backend/models/best_stage3_new.pt')
    print('  3. Restart backend')
else:
    # Lokal: salin langsung ke backend/models
    models_dir = Path('F:/migas-siews/backend/models')
    models_dir.mkdir(parents=True, exist_ok=True)
    out_best = models_dir / 'best_stage3_new.pt'
    export_model.save(str(out_best))
    print(f'✅ Saved to backend: {out_best}  ({out_best.stat().st_size/1e6:.1f} MB)')

## 12. Saran jika mAP Rendah

| Masalah | Solusi |
|---|---|
| mAP50 < 0.70 | Tambah data dari Roboflow: cari `fire detection` / `smoke detection` |
| Banyak false positive | Tambah gambar negatif (tanpa api/asap) ke dataset |
| Detection NONE saat test | Turunkan `conf` threshold ke 0.15 |
| Training lambat | Kurangi `batch=8`, atau upgrade ke GPU T4 x2 di Kaggle |
| Loss tidak turun | Kurangi `lr0=0.0005`, naikkan `warmup_epochs=5` |
| Overfitting | Naikkan `weight_decay=0.001`, tambah `dropout=0.1` |

### Dataset tambahan yang recommended (gratis di Roboflow):
- `D-Fire Dataset` — 21k gambar api & asap outdoor
- `Fire Detection v2` — 5k gambar indoor & outdoor  
- `Wildfire Smoke` — khusus asap hutan